In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  FL-BASED SMART AGRICULTURE — SOYBEAN DISEASE DETECTION
#  Model: ResNet-18 | Dataset: Soybean Diseased Leaf (Kaggle)
#  Algorithms: Centralized, FedAvg, FedSGD, FedProx, FedAdam, FedDyn
#  Client Configs: 3-client × 8 rounds, 9-client × 8 rounds
# ═══════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ──────────────────────────────────────────────────────────────────
import sys

!{sys.executable} -m pip install "numpy==1.26.4" --force-reinstall -q
!{sys.executable} -m pip install torch==2.4.0 torchvision==0.19.0 -q
!{sys.executable} -m pip install matplotlib seaborn scikit-learn tqdm -q

print("✅ All packages installed!")
print("ACTION: Runtime > Restart Session — then run Cell 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 2 — Imports & device setup
# ──────────────────────────────────────────────────────────────────
import numpy as np
print(f"Numpy   : {np.__version__}")

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import copy, random, os, json
from tqdm import tqdm

print(f"Torch   : {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("⚠  No GPU — consider enabling T4 GPU in Runtime > Change runtime type")

Numpy   : 1.26.4
Torch   : 2.4.0+cu121
Device  : cuda
GPU     : Tesla T4


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 3 — Kaggle credentials & dataset download
# ──────────────────────────────────────────────────────────────────
from google.colab import files

print("📁 Upload your kaggle.json file:")
print("   Get it from: kaggle.com → Account → Settings → API → Create New Token")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ kaggle.json saved!")

# Download Soybean Diseased Leaf Dataset
# Dataset: https://www.kaggle.com/datasets/sivm205/soybean-diseased-leaf-dataset
!kaggle datasets download -d sivm205/soybean-diseased-leaf-dataset --unzip -p ./data
print("\n✅ Download complete!")
print(os.listdir('./data'))

📁 Upload your kaggle.json file:
   Get it from: kaggle.com → Account → Settings → API → Create New Token


Saving kaggle.json to kaggle.json
✅ kaggle.json saved!
Dataset URL: https://www.kaggle.com/datasets/sivm205/soybean-diseased-leaf-dataset
License(s): unknown
100% 1.93G/1.93G [00:24<00:00, 85.9MB/s]


✅ Download complete!
['Sudden Death Syndrone', 'powdery_mildew', 'bacterial_blight', 'Yellow Mosaic', 'Mossaic Virus', 'crestamento', 'septoria', 'brown_spot', 'ferrugen', 'Southern blight']


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 4 — Explore folder structure
# ──────────────────────────────────────────────────────────────────
print("📂 Folder structure inside ./data:\n")
for item in sorted(os.listdir('./data')):
    full_path = os.path.join('./data', item)
    if os.path.isdir(full_path):
        sub = os.listdir(full_path)
        print(f"  FOLDER : ./data/{item}")
        print(f"  Classes: {len(sub)}")
        print(f"  Names  : {sub}")
        print()


📂 Folder structure inside ./data:

  FOLDER : ./data/Mossaic Virus
  Classes: 22
  Names  : ['DSC_0138.jpg', 'DSC_0127.jpg', 'DSC_0136.jpg', 'DSC_0144.jpg', 'DSC_0114.jpg', 'DSC_0140.jpg', 'DSC_0145.jpg', 'DSC_0139.jpg', 'DSC_0141.jpg', 'DSC_0147.jpg', 'DSC_0143.jpg', 'DSC_0134.jpg', 'DSC_0132.jpg', 'DSC_0135.jpg', 'DSC_0137.jpg', 'DSC_0133.jpg', 'DSC_0129.jpg', 'DSC_0146.jpg', 'DSC_0130.jpg', 'DSC_0128.jpg', 'DSC_0131.jpg', 'DSC_0142.jpg']

  FOLDER : ./data/Southern blight
  Classes: 62
  Names  : ['DSC_0047.jpg', 'DSC_0032.jpg', 'DSC_0174.jpg', 'DSC_0108.jpg', 'DSC_0027.jpg', 'DSC_0031.jpg', 'DSC_0041.jpg', 'DSC_0104.jpg', 'DSC_0150.jpg', 'DSC_0106.jpg', 'DSC_0043.jpg', 'DSC_0026.jpg', 'DSC_0051 (2).jpg', 'DSC_0025.jpg', 'DSC_0051.jpg', 'DSC_0029 (2).jpg', 'DSC_0029.jpg', 'DSC_0102.jpg', 'DSC_0099.jpg', 'DSC_0097.jpg', 'DSC_0107.jpg', 'DSC_0054.jpg', 'DSC_0023.jpg', 'DSC_0171.jpg', 'DSC_0045.jpg', 'DSC_0037.jpg', 'DSC_0042.jpg', 'DSC_0040.jpg', 'DSC_0096.jpg', 'DSC_0030.jpg', 'DSC_0

In [ ]:
# CELL 5 — FIXED
import os, random
import numpy as np
import torch
from torchvision import datasets, transforms

SEED       = 42
BATCH_SIZE = 32
IMG_SIZE   = 224

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = './data'   # ← fixed

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)

print(f"✅ Dataset loaded!")
print(f"   Total images  : {len(full_dataset)}")
print(f"   Total classes : {NUM_CLASSES}")
print(f"   Classes       : {CLASS_NAMES}")

✅ Dataset loaded!
   Total images  : 701
   Total classes : 10
   Classes       : ['Mossaic Virus', 'Southern blight', 'Sudden Death Syndrone', 'Yellow Mosaic', 'bacterial_blight', 'brown_spot', 'crestamento', 'ferrugen', 'powdery_mildew', 'septoria']


In [ ]:
# CELL 5B — class distribution check
from collections import Counter

label_counts = Counter([label for _, label in full_dataset.samples])
print("Class distribution:")
for idx, name in enumerate(CLASS_NAMES):
    count = label_counts[idx]
    flag  = "⚠️  VERY SMALL" if count < 20 else ""
    print(f"  {name:<30} : {count:>4} images  {flag}")

total_imgs = len(full_dataset)
print(f"\nTotal: {total_imgs} images | {NUM_CLASSES} classes")
print(f"\nWith 3 clients, each gets ~{int(total_imgs * 0.8 // 3)} training samples")
print(f"With 9 clients, each gets ~{int(total_imgs * 0.8 // 9)} training samples")

Class distribution:
  Mossaic Virus                  :   22 images  
  Southern blight                :   62 images  
  Sudden Death Syndrone          :  110 images  
  Yellow Mosaic                  :  110 images  
  bacterial_blight               :   88 images  
  brown_spot                     :   81 images  
  crestamento                    :    5 images  ⚠️  VERY SMALL
  ferrugen                       :   65 images  
  powdery_mildew                 :  137 images  
  septoria                       :   21 images  

Total: 701 images | 10 classes

With 3 clients, each gets ~186 training samples
With 9 clients, each gets ~62 training samples


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 6 — Train/test split + client partitioning helper
# ──────────────────────────────────────────────────────────────────
total   = len(full_dataset)
indices = list(range(total))
random.shuffle(indices)

split     = int(0.8 * total)
train_idx = indices[:split]
test_idx  = indices[split:]

# Train subset
train_set = Subset(full_dataset, train_idx)

# Test subset (uses test_transform)
test_dataset = datasets.ImageFolder(root=DATA_DIR, transform=test_transform)
test_set     = Subset(test_dataset, test_idx)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

def create_client_datasets(train_set, num_clients, seed=SEED):
    """IID partition of training data across num_clients."""
    rng = random.Random(seed)
    n   = len(train_set)
    idx = list(range(n))
    rng.shuffle(idx)
    splits = np.array_split(idx, num_clients)
    return [Subset(train_set, s.tolist()) for s in splits]

def get_client_loaders(client_datasets):
    return [DataLoader(ds, batch_size=BATCH_SIZE,
                       shuffle=True, num_workers=2)
            for ds in client_datasets]

print(f"✅ Train : {len(train_set)} samples")
print(f"   Test  : {len(test_set)} samples")

✅ Train : 560 samples
   Test  : 141 samples


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 7 — Model definition (ResNet-18, transfer learning)
# ──────────────────────────────────────────────────────────────────
def get_model(num_classes=NUM_CLASSES):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze last residual block (layer4)
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Custom classifier head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model

# Quick sanity check
_m   = get_model().to(device)
_out = _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device))
_tp  = sum(p.numel() for p in _m.parameters())
_tr  = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"✅ Model OK  |  Output: {_out.shape}  |  "
      f"Trainable: {_tr:,} / {_tp:,}")
del _m, _out
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 84.5MB/s]


✅ Model OK  |  Output: torch.Size([2, 10])  |  Trainable: 8,528,138 / 11,310,922


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 8 — Local training & evaluation functions
# ──────────────────────────────────────────────────────────────────
def local_train(model, dataloader, epochs=3, lr=0.001,
                mu=0.0, global_weights=None):
    model     = model.to(device).train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    last_acc = 0.0
    for epoch in range(epochs):
        running_loss = correct = total = 0
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)

            # FedProx proximal term
            if mu > 0.0 and global_weights is not None:
                prox = sum(((p - global_weights[n].to(device))**2).sum()
                           for n, p in model.named_parameters()
                           if p.requires_grad)
                loss += (mu / 2.0) * prox

            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, pred  = outputs.max(1)
            correct  += pred.eq(labels).sum().item()
            total    += labels.size(0)

        scheduler.step()
        last_acc = 100.0 * correct / total
        print(f"    Epoch [{epoch+1}/{epochs}]  "
              f"Loss: {running_loss/len(dataloader):.4f}  "
              f"Acc: {last_acc:.2f}%")

    return model.state_dict(), last_acc


def evaluate(model, dataloader):
    model = model.to(device).eval()
    correct = total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred  = outputs.max(1)
            correct += pred.eq(labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = 100.0 * correct / total
    return acc, all_preds, all_labels

print("✅ local_train() & evaluate() ready")

✅ local_train() & evaluate() ready


In [ ]:
# CELL 9 — Aggregation algorithms (FIXED)
def fed_avg(global_w, client_w_list, client_sizes):
    total = sum(client_sizes)
    new_w = copy.deepcopy(global_w)
    for key in new_w:
        new_w[key] = sum((client_sizes[i] / total) * cw[key].float()
                         for i, cw in enumerate(client_w_list))
    return new_w

def fed_sgd(global_w, client_w_list, client_sizes, lr=0.1):
    total = sum(client_sizes)
    new_w = copy.deepcopy(global_w)
    for key in new_w:
        grad = sum((client_sizes[i] / total) *
                   (global_w[key].float() - cw[key].float())
                   for i, cw in enumerate(client_w_list))
        new_w[key] = global_w[key].float() - lr * grad
    return new_w

def fed_prox(global_w, client_w_list, client_sizes):
    return fed_avg(global_w, client_w_list, client_sizes)

def fed_adam(global_w, client_w_list, client_sizes,
             state, lr=0.0001, b1=0.9, b2=0.999, eps=1e-3):
    total = sum(client_sizes)
    if state['m'] is None:
        state['m'] = {k: torch.zeros_like(v).float() for k, v in global_w.items()}
        state['v'] = {k: torch.zeros_like(v).float() for k, v in global_w.items()}

    delta = {key: sum((client_sizes[i] / total) *
                      (global_w[key].float() - cw[key].float())
                      for i, cw in enumerate(client_w_list))
             for key in global_w}

    new_w = copy.deepcopy(global_w)
    state['t'] += 1
    for key in new_w:
        state['m'][key] = b1 * state['m'][key] + (1 - b1) * delta[key]
        state['v'][key] = b2 * state['v'][key] + (1 - b2) * delta[key]**2
        m_hat = state['m'][key] / (1 - b1**state['t'])
        v_hat = state['v'][key] / (1 - b2**state['t'])
        new_w[key] = global_w[key].float() + lr * m_hat / (v_hat.sqrt() + eps)
    return new_w, state

def fed_dyn(global_w, client_w_list, client_sizes, h_state, alpha=0.01):
    total = sum(client_sizes)
    avg_w = {key: sum((client_sizes[i] / total) * cw[key].float()
                      for i, cw in enumerate(client_w_list))
             for key in global_w}

    if h_state['h'] is None:
        h_state['h'] = {k: torch.zeros_like(v).float()  # ✅ FIXED
                        for k, v in global_w.items()}

    for key in global_w:
        h_state['h'][key] -= alpha * (avg_w[key] - global_w[key].float())

    new_w = {key: avg_w[key] - (1.0 / alpha) * h_state['h'][key]
             for key in global_w}
    return new_w, h_state

print("✅ FedAvg | FedSGD | FedProx | FedAdam | FedDyn — all ready")

✅ FedAvg | FedSGD | FedProx | FedAdam | FedDyn — all ready


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 10 — Centralized baseline training
# ──────────────────────────────────────────────────────────────────
def centralized_train(num_epochs=10, lr=0.001):
    print(f"\n{'='*58}")
    print(f"  CENTRALIZED BASELINE — ResNet-18")
    print(f"  Dataset : Soybean Disease Detection")
    print(f"  Epochs  : {num_epochs}")
    print(f"{'='*58}\n")

    # Full training loader (all training data, no partition)
    full_train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=2)

    model     = get_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    epoch_accs = []
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = correct = total = 0
        for images, labels in full_train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, pred  = outputs.max(1)
            correct  += pred.eq(labels).sum().item()
            total    += labels.size(0)

        scheduler.step()
        acc, _, _ = evaluate(model, test_loader)
        epoch_accs.append(acc)
        print(f"  Epoch [{epoch:>2}/{num_epochs}]  "
              f"Loss: {running_loss/len(full_train_loader):.4f}  "
              f"Test Acc: {acc:.2f}%")
        torch.cuda.empty_cache()

    final_acc, preds, labels_gt = evaluate(model, test_loader)
    print(f"\n{'='*58}")
    print(f"  CENTRALIZED — Final Acc : {final_acc:.2f}%")
    print(f"  CENTRALIZED — Best Acc  : {max(epoch_accs):.2f}%")
    print(f"{'='*58}")
    return model, epoch_accs, preds, labels_gt

print("✅ centralized_train() ready")

✅ centralized_train() ready


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 11 — Federated Learning training loop
# ──────────────────────────────────────────────────────────────────
def fl_train(method='fedavg', num_rounds=8, local_epochs=3,
             lr=0.001, mu=0.01, num_clients=5):

    print(f"\n{'='*62}")
    print(f"  FL-Based Smart Agriculture — Soybean Disease Detection")
    print(f"  Model   : ResNet-18")
    print(f"  Method  : {method.upper()}")
    print(f"  Clients : {num_clients}  |  Rounds: {num_rounds}  |  Epochs: {local_epochs}")
    print(f"{'='*62}\n")

    # Build client data for this run
    c_datasets = create_client_datasets(train_set, num_clients)
    c_loaders  = get_client_loaders(c_datasets)
    c_sizes    = [len(ds) for ds in c_datasets]

    global_model   = get_model(NUM_CLASSES).to(device)
    global_weights = copy.deepcopy(global_model.state_dict())
    adam_state     = {'m': None, 'v': None, 't': 0}
    dyn_state      = {'h': None}
    round_accs     = []

    for rnd in range(1, num_rounds + 1):
        print(f"\n{'─'*50}")
        print(f"  ROUND {rnd}/{num_rounds} — {method.upper()} | {num_clients} clients")
        print(f"{'─'*50}")

        client_weights = []
        local_accs     = []

        for cid in range(num_clients):
            print(f"\n  [Client {cid+1}/{num_clients}] {c_sizes[cid]} samples")
            local_model = get_model(NUM_CLASSES)
            local_model.load_state_dict(copy.deepcopy(global_weights))

            mu_val      = mu if method == 'fedprox' else 0.0
            gw_for_prox = global_weights if method == 'fedprox' else None

            updated_w, lacc = local_train(
                local_model, c_loaders[cid],
                epochs=local_epochs, lr=lr,
                mu=mu_val, global_weights=gw_for_prox
            )
            client_weights.append(updated_w)
            local_accs.append(lacc)

        print(f"\n  [Server] Aggregating — {method.upper()}")
        if method == 'fedavg':
            global_weights = fed_avg(global_weights, client_weights, c_sizes)
        elif method == 'fedsgd':
            global_weights = fed_sgd(global_weights, client_weights, c_sizes)
        elif method == 'fedprox':
            global_weights = fed_prox(global_weights, client_weights, c_sizes)
        elif method == 'fedadam':
            global_weights, adam_state = fed_adam(
                global_weights, client_weights, c_sizes, adam_state)
        elif method == 'feddyn':
            global_weights, dyn_state = fed_dyn(
                global_weights, client_weights, c_sizes, dyn_state, alpha=0.01)

        global_model.load_state_dict(global_weights)
        acc, _, _ = evaluate(global_model, test_loader)
        round_accs.append(acc)
        avg_local = sum(local_accs) / len(local_accs)

        print(f"\n  ✅ [Round {rnd}]  Global: {acc:.2f}%  |  Avg Local: {avg_local:.2f}%")
        torch.cuda.empty_cache()

    print(f"\n{'='*62}")
    print(f"  DONE — {method.upper()} | {num_clients} clients")
    print(f"  Final Acc : {round_accs[-1]:.2f}%")
    print(f"  Best Acc  : {max(round_accs):.2f}% (Round {round_accs.index(max(round_accs))+1})")
    print(f"{'='*62}")
    return global_model, round_accs

print("✅ fl_train() ready")

✅ fl_train() ready


In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL RESUME CELL — Only fedprox_9c, fedadam_9c, feddyn_9c left
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# ── Hyperparameters (must define since Cell 12 was skipped) ───────
NUM_ROUNDS   = 8
LOCAL_EPOCHS = 3
LR           = 0.001
FL_METHODS   = ['fedavg', 'fedsgd', 'fedprox', 'fedadam', 'feddyn']

# ── Reload all results from Drive ────────────────────────────────
with open('/content/drive/MyDrive/FL_Soybean/results_so_far.json', 'r') as f:
    all_results = json.load(f)

print("✅ Results loaded. Completed so far:")
for key, accs in all_results.items():
    print(f"   {key:<22} Final: {accs[-1]:>6.2f}%  Best: {max(accs):>6.2f}%")

# ── Reload all saved models from Drive ───────────────────────────
all_models = {}
for key in all_results.keys():
    path = f'/content/drive/MyDrive/FL_Soybean/model_{key}.pth'
    if os.path.exists(path):
        m = get_model(NUM_CLASSES).to(device)
        m.load_state_dict(torch.load(
            path, map_location=device, weights_only=False))
        all_models[key] = m
        print(f"✅ Model loaded: {key}")
    else:
        print(f"⚠️  Not found: {path}")

# ── Run only the 3 remaining experiments ─────────────────────────
REMAINING = [
    ('fedprox', 9),
    ('fedadam', 9),
    ('feddyn',  9),
]

for method, n_clients in REMAINING:
    exp_key = f"{method}_{n_clients}c"

    if exp_key in all_results:
        print(f"\n⏭  Skipping {exp_key} — already done")
        continue

    print(f"\n{'#'*62}")
    print(f"  RUNNING: {exp_key.upper()}")
    print(f"{'#'*62}")

    model, accs = fl_train(
        method=method, num_rounds=NUM_ROUNDS,
        local_epochs=LOCAL_EPOCHS, lr=LR,
        mu=0.01, num_clients=n_clients
    )
    all_results[exp_key] = accs
    all_models[exp_key]  = model

    torch.save(model.state_dict(),
               f'/content/drive/MyDrive/FL_Soybean/model_{exp_key}.pth')
    with open('/content/drive/MyDrive/FL_Soybean/results_so_far.json', 'w') as f:
        json.dump(all_results, f)
    print(f"✅ Saved: {exp_key}")
    torch.cuda.empty_cache()

print("\n\n✅✅ ALL TRAINING COMPLETE ✅✅")
print("\nFinal Results:")
print("="*58)
for key, accs in all_results.items():
    print(f"  {key:<22} Final: {accs[-1]:>6.2f}%  Best: {max(accs):>6.2f}%")
print("="*58)

/tmp/ipykernel_3735/724562397.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(


✅ Loaded: centralized  |  Best acc: 99.29%
✅ Loaded: fedavg_3c  |  Best acc: 99.29%
✅ Loaded: fedsgd_3c  |  Best acc: 91.49%
✅ Loaded: fedprox_3c  |  Best acc: 98.58%

##############################################################
  RUNNING: FEDDYN_3C
##############################################################

  FL-Based Smart Agriculture — Soybean Disease Detection
  Model   : ResNet-18
  Method  : FEDDYN
  Clients : 3  |  Rounds: 8  |  Epochs: 3


──────────────────────────────────────────────────
  ROUND 1/8 — FEDDYN | 3 clients
──────────────────────────────────────────────────

  [Client 1/3] 187 samples
    Epoch [1/3]  Loss: 1.3317  Acc: 56.68%
    Epoch [2/3]  Loss: 0.5013  Acc: 86.63%
    Epoch [3/3]  Loss: 0.3576  Acc: 90.37%

  [Client 2/3] 187 samples
    Epoch [1/3]  Loss: 1.2751  Acc: 60.96%
    Epoch [2/3]  Loss: 0.5451  Acc: 84.49%
    Epoch [3/3]  Loss: 0.3079  Acc: 93.58%

  [Client 3/3] 186 samples
    Epoch [1/3]  Loss: 1.2323  Acc: 63.44%
    Epoch [2/3]  Loss:

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RELOAD CELL — Load all saved results and models from Drive
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# Reload results
with open('/content/drive/MyDrive/FL_Soybean/results_so_far.json', 'r') as f:
    all_results = json.load(f)

# Reload all models
all_models = {}
for key in all_results.keys():
    path = f'/content/drive/MyDrive/FL_Soybean/model_{key}.pth'
    if os.path.exists(path):
        m = get_model(NUM_CLASSES).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        all_models[key] = m
        print(f"✅ Loaded: {key:<22} Final: {all_results[key][-1]:>6.2f}%  Best: {max(all_results[key]):>6.2f}%")
    else:
        print(f"⚠️  Model file not found: {path}")

print(f"\n✅ Total loaded: {len(all_models)} models")
print(f"✅ Total results: {len(all_results)} experiments")

# These are needed for Cell 17 (confusion matrix)
best_fl_key = max(
    [k for k in all_results if k != 'centralized'],
    key=lambda k: all_results[k][-1]
)
best_model = all_models[best_fl_key]
acc, cent_preds, cent_labels = evaluate(all_models['centralized'], test_loader)
acc2, preds, labels_gt = evaluate(best_model, test_loader)

print(f"\n🏆 Best FL model : {best_fl_key.upper()}")
print(f"   Accuracy      : {acc2:.2f}%")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 12 — Mount Drive & run all experiments
# ──────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/FL_Soybean', exist_ok=True)
print("✅ Drive mounted → /content/drive/MyDrive/FL_Soybean/")

# ── Hyperparameters ───────────────────────────────────────────────
NUM_ROUNDS   = 8   # rounds per FL experiment (matches your notes: 8)
LOCAL_EPOCHS = 3
LR           = 0.001

# ── Experiment registry ───────────────────────────────────────────
all_results = {}   # key → list of per-round accuracies
all_models  = {}   # key → trained model

FL_METHODS  = ['fedavg', 'fedsgd', 'fedprox', 'fedadam', 'feddyn']
CLIENT_CONFIGS = [3, 9]    # your notes: 3 clients & 9 clients

# ═══ Step A: Centralized baseline ════════════════════════════════
cent_model, cent_accs, cent_preds, cent_labels = centralized_train(
    num_epochs=NUM_ROUNDS, lr=LR
)
all_results['centralized'] = cent_accs
all_models['centralized']  = cent_model

torch.save(cent_model.state_dict(),
           '/content/drive/MyDrive/FL_Soybean/model_centralized.pth')
with open('/content/drive/MyDrive/FL_Soybean/results_so_far.json', 'w') as f:
    json.dump(all_results, f)
print("✅ Centralized checkpoint saved")
torch.cuda.empty_cache()

# ═══ Step B: FL experiments (3-client & 9-client) ════════════════
for n_clients in CLIENT_CONFIGS:
    for method in FL_METHODS:
        exp_key = f"{method}_{n_clients}c"
        print(f"\n\n{'#'*62}")
        print(f"  EXPERIMENT: {exp_key.upper()}")
        print(f"{'#'*62}")

        model, accs = fl_train(
            method      = method,
            num_rounds  = NUM_ROUNDS,
            local_epochs= LOCAL_EPOCHS,
            lr          = LR,
            mu          = 0.01,
            num_clients = n_clients
        )
        all_results[exp_key] = accs
        all_models[exp_key]  = model

        # Save after every experiment
        torch.save(model.state_dict(),
                   f'/content/drive/MyDrive/FL_Soybean/model_{exp_key}.pth')
        with open('/content/drive/MyDrive/FL_Soybean/results_so_far.json', 'w') as f:
            json.dump(all_results, f)
        print(f"✅ Checkpoint saved: {exp_key}")
        torch.cuda.empty_cache()

print("\n\n✅✅ ALL EXPERIMENTS COMPLETE ✅✅")

Mounted at /content/drive
✅ Drive mounted → /content/drive/MyDrive/FL_Soybean/

  CENTRALIZED BASELINE — ResNet-18
  Dataset : Soybean Disease Detection
  Epochs  : 8

  Epoch [ 1/8]  Loss: 0.7310  Test Acc: 87.94%
  Epoch [ 2/8]  Loss: 0.3214  Test Acc: 88.65%
  Epoch [ 3/8]  Loss: 0.2780  Test Acc: 93.62%
  Epoch [ 4/8]  Loss: 0.1932  Test Acc: 92.20%
  Epoch [ 5/8]  Loss: 0.1456  Test Acc: 99.29%
  Epoch [ 6/8]  Loss: 0.0898  Test Acc: 95.74%
  Epoch [ 7/8]  Loss: 0.0834  Test Acc: 97.87%
  Epoch [ 8/8]  Loss: 0.1008  Test Acc: 99.29%

  CENTRALIZED — Final Acc : 99.29%
  CENTRALIZED — Best Acc  : 99.29%
✅ Centralized checkpoint saved


##############################################################
  EXPERIMENT: FEDAVG_3C
##############################################################

  FL-Based Smart Agriculture — Soybean Disease Detection
  Model   : ResNet-18
  Method  : FEDAVG
  Clients : 3  |  Rounds: 8  |  Epochs: 3


──────────────────────────────────────────────────
  ROUND 

RuntimeError: result type Float can't be cast to the desired output type Long

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 13 — Summary table
# ──────────────────────────────────────────────────────────────────
print("\nFULL RESULTS SUMMARY")
print("="*72)
print(f"{'Experiment':<22} {'Round 1':>9} {'Final':>9} {'Best':>9} {'Best Rnd':>10}")
print("-"*72)

for key, accs in all_results.items():
    if accs:
        print(f"{key.upper():<22} "
              f"{accs[0]:>8.2f}% "
              f"{accs[-1]:>8.2f}% "
              f"{max(accs):>8.2f}% "
              f"{accs.index(max(accs))+1:>9}")
print("="*72)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 14 — Plot 1: Centralized vs Federated (3-client) comparison
# ──────────────────────────────────────────────────────────────────
colors = {
    'centralized': '#222222',
    'fedavg':      '#378ADD',
    'fedsgd':      '#D85A30',
    'fedprox':     '#1D9E75',
    'fedadam':     '#BA7517',
    'feddyn':      '#7F77DD'
}
markers = {
    'centralized': '*',
    'fedavg':      'o',
    'fedsgd':      's',
    'fedprox':     '^',
    'fedadam':     'D',
    'feddyn':      'P'
}

def plot_comparison(client_config, title_suffix, filename):
    fig, ax = plt.subplots(figsize=(13, 6))
    rounds  = range(1, NUM_ROUNDS + 1)

    # Centralized
    ax.plot(rounds, all_results['centralized'],
            marker=markers['centralized'], color=colors['centralized'],
            label='CENTRALIZED (Baseline)', linewidth=2.8,
            markersize=9, linestyle='--', zorder=5)

    for method in FL_METHODS:
        key = f"{method}_{client_config}c"
        if key in all_results:
            ax.plot(rounds, all_results[key],
                    marker=markers[method], color=colors[method],
                    label=f"{method.upper()} ({client_config} clients)",
                    linewidth=2.5, markersize=7)

    ax.set_xlabel('Communication Round / Epoch', fontsize=13)
    ax.set_ylabel('Global Test Accuracy (%)', fontsize=13)
    ax.set_title(
        f'Centralized vs Federated — {title_suffix}\n'
        'Soybean Disease Detection | ResNet-18',
        fontsize=14, fontweight='bold'
    )
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.25)
    ax.set_xticks(list(rounds))
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {filename}")

plot_comparison(3, '3 Clients × 8 Rounds', 'fl_comparison_3clients.png')
plot_comparison(9, '9 Clients × 8 Rounds', 'fl_comparison_9clients.png')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 15 — Plot 2: 3-client vs 9-client (per algorithm)
# ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)
axes = axes.flatten()
rounds = range(1, NUM_ROUNDS + 1)

# Centralized in first subplot
ax = axes[0]
ax.plot(rounds, all_results['centralized'],
        color=colors['centralized'], marker='*',
        linewidth=2.5, markersize=9, label='Centralized')
ax.set_title('CENTRALIZED (Baseline)', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.grid(alpha=0.25)

for i, method in enumerate(FL_METHODS):
    ax  = axes[i + 1]
    k3  = f"{method}_3c"
    k9  = f"{method}_9c"
    if k3 in all_results:
        ax.plot(rounds, all_results[k3],
                color='#378ADD', marker='o', linewidth=2.5,
                markersize=7, label='3 clients')
    if k9 in all_results:
        ax.plot(rounds, all_results[k9],
                color='#D85A30', marker='s', linewidth=2.5,
                markersize=7, label='9 clients')
    ax.set_title(f'{method.upper()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(alpha=0.25)
    ax.set_xticks(list(rounds))

plt.suptitle(
    '3-Client vs 9-Client Comparison — Soybean Disease Detection\n'
    'Federated Learning with ResNet-18',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('fl_3vs9_clients.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fl_3vs9_clients.png")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 16 — Plot 3: FL algorithms head-to-head (3c vs 9c side by side)
# ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for method in FL_METHODS:
    c  = colors[method]; mk = markers[method]
    k3 = f"{method}_3c"; k9 = f"{method}_9c"
    if k3 in all_results:
        ax1.plot(rounds, all_results[k3], color=c, marker=mk,
                 linewidth=2.5, markersize=7, label=method.upper())
    if k9 in all_results:
        ax2.plot(rounds, all_results[k9], color=c, marker=mk,
                 linewidth=2.5, markersize=7, label=method.upper())

for ax, title in [(ax1, '3-Client Setup (8 Rounds)'),
                  (ax2, '9-Client Setup (8 Rounds)')]:
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Communication Round', fontsize=12)
    ax.set_ylabel('Global Test Accuracy (%)', fontsize=12)
    ax.legend(fontsize=11); ax.grid(alpha=0.25)
    ax.set_xticks(list(rounds))

plt.suptitle(
    'FL Algorithm Comparison — Soybean Disease Detection | ResNet-18',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fl_algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fl_algorithm_comparison.png")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 17 — Confusion matrix & F1 score for best FL model
# ──────────────────────────────────────────────────────────────────

# Pick best FL model across all experiments (excluding centralized)
fl_keys    = [k for k in all_results if k != 'centralized' and all_results[k]]
best_key   = max(fl_keys, key=lambda k: all_results[k][-1])
best_model = all_models[best_key]

acc, preds, labels_gt = evaluate(best_model, test_loader)
print(f"\n🏆 Best FL Model : {best_key.upper()}")
print(f"   Test Accuracy : {acc:.2f}%")

# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(labels_gt, preds)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES + 2),
                                max(7, NUM_CLASSES + 1)))
sns.heatmap(cm, annot=True if NUM_CLASSES <= 10 else False,
            fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, ax=ax)
ax.set_title(
    f'Confusion Matrix — {best_key.upper()}\nSoybean Disease Detection',
    fontsize=13, fontweight='bold'
)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix_best_fl.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: confusion_matrix_best_fl.png")

# ── Centralized confusion matrix ──────────────────────────────────
cm_c = confusion_matrix(cent_labels, cent_preds)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES + 2),
                                max(7, NUM_CLASSES + 1)))
sns.heatmap(cm_c, annot=True if NUM_CLASSES <= 10 else False,
            fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, ax=ax)
ax.set_title(
    'Confusion Matrix — CENTRALIZED (Baseline)\nSoybean Disease Detection',
    fontsize=13, fontweight='bold'
)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix_centralized.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: confusion_matrix_centralized.png")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 18 — F1 score bar chart (best FL model)
# ──────────────────────────────────────────────────────────────────
report   = classification_report(labels_gt, preds,
                                  target_names=CLASS_NAMES,
                                  output_dict=True)
f1_data  = {cls: report[cls]['f1-score'] for cls in CLASS_NAMES}
sorted_f1 = sorted(f1_data.items(), key=lambda x: x[1])
cls_names = [x[0] for x in sorted_f1]
f1_vals   = [x[1] for x in sorted_f1]
bar_colors = ['#D85A30' if v < 0.60 else
              '#BA7517' if v < 0.80 else
              '#1D9E75' for v in f1_vals]

fig, ax = plt.subplots(figsize=(10, max(5, NUM_CLASSES * 0.5 + 2)))
bars = ax.barh(cls_names, f1_vals, color=bar_colors,
               edgecolor='white', linewidth=0.5)
ax.set_xlabel('F1 Score', fontsize=12)
ax.set_title(
    f'Per-Class F1 Score — {best_key.upper()}\nSoybean Disease Detection',
    fontsize=13, fontweight='bold'
)
ax.axvline(x=0.60, color='#D85A30', linestyle='--', alpha=0.6, label='< 0.60')
ax.axvline(x=0.80, color='#BA7517', linestyle='--', alpha=0.6, label='< 0.80')
ax.legend(fontsize=10)
ax.set_xlim(0, 1.1)
for bar, val in zip(bars, f1_vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('f1_scores_best_fl.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: f1_scores_best_fl.png")

# Classification report
print("\nCLASSIFICATION REPORT — Best FL Model")
print("="*60)
print(classification_report(labels_gt, preds, target_names=CLASS_NAMES))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 19 — Bar chart: Final accuracy comparison (all experiments)
# ──────────────────────────────────────────────────────────────────
exp_labels = list(all_results.keys())
final_accs = [all_results[k][-1] for k in exp_labels]
bar_c = ['#222222' if k == 'centralized'
         else colors.get(k.split('_')[0], '#888888')
         for k in exp_labels]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(exp_labels, final_accs, color=bar_c, edgecolor='white',
              linewidth=0.8, width=0.65)
ax.set_xlabel('Experiment', fontsize=12)
ax.set_ylabel('Final Test Accuracy (%)', fontsize=12)
ax.set_title(
    'Final Accuracy — All Experiments\nSoybean Disease Detection | ResNet-18',
    fontsize=13, fontweight='bold'
)
ax.set_ylim(0, 105)
plt.xticks(rotation=35, ha='right', fontsize=9)
for bar, val in zip(bars, final_accs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8.5)

# Legend
legend_patches = [
    mpatches.Patch(color='#222222', label='Centralized'),
    mpatches.Patch(color='#378ADD', label='FedAvg'),
    mpatches.Patch(color='#D85A30', label='FedSGD'),
    mpatches.Patch(color='#1D9E75', label='FedProx'),
    mpatches.Patch(color='#BA7517', label='FedAdam'),
    mpatches.Patch(color='#7F77DD', label='FedDyn'),
]
ax.legend(handles=legend_patches, fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('final_accuracy_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: final_accuracy_bar.png")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 20 — Save results & copy everything to Drive
# ──────────────────────────────────────────────────────────────────
with open('results_summary.txt', 'w') as f:
    f.write("FL-BASED SMART AGRICULTURE — SOYBEAN DISEASE DETECTION\n")
    f.write("="*70 + "\n")
    f.write(f"Dataset      : Soybean Diseased Leaf (Kaggle: sivm205)\n")
    f.write(f"Model        : ResNet-18 (Transfer Learning)\n")
    f.write(f"Classes      : {NUM_CLASSES}\n")
    f.write(f"Rounds/Epochs: {NUM_ROUNDS}\n")
    f.write(f"Local Epochs : {LOCAL_EPOCHS}\n\n")
    f.write(f"{'Experiment':<22} {'Final Acc':>10} {'Best Acc':>10} {'Best Rnd':>10}\n")
    f.write("-"*55 + "\n")
    for k, accs in all_results.items():
        if accs:
            f.write(f"{k.upper():<22} "
                    f"{accs[-1]:>9.2f}% "
                    f"{max(accs):>9.2f}% "
                    f"{accs.index(max(accs))+1:>9}\n")
    f.write("\n\nClassification Report (Best FL Model):\n")
    f.write(classification_report(labels_gt, preds, target_names=CLASS_NAMES))

print("✅ results_summary.txt saved")

# Copy all outputs to Drive
output_files = [
    'fl_comparison_3clients.png',
    'fl_comparison_9clients.png',
    'fl_3vs9_clients.png',
    'fl_algorithm_comparison.png',
    'confusion_matrix_best_fl.png',
    'confusion_matrix_centralized.png',
    'f1_scores_best_fl.png',
    'final_accuracy_bar.png',
    'results_summary.txt',
]
for fname in output_files:
    if os.path.exists(fname):
        !cp {fname} /content/drive/MyDrive/FL_Soybean/
        print(f"  📁 Copied: {fname}")

!ls /content/drive/MyDrive/FL_Soybean/
print("\n✅ PROJECT COMPLETE! All files saved to Drive.")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 21 — Image inference (test any leaf image)
# ──────────────────────────────────────────────────────────────────
from google.colab import files
from PIL import Image
import torch.nn.functional as F

assert 'CLASS_NAMES' in dir() and len(CLASS_NAMES) > 0, \
    "Re-run dataset loading cell first!"
print(f"Classes: {CLASS_NAMES}")

# Build clean inference model
best_fl_key   = max(
    [k for k in all_results if k != 'centralized'],
    key=lambda k: all_results[k][-1]
)
infer_model = get_model(NUM_CLASSES)
infer_model.load_state_dict(
    copy.deepcopy(all_models[best_fl_key].state_dict()))
infer_model = infer_model.to(device).eval()
print(f"✅ Inference model ready: {best_fl_key.upper()}")

# Upload image
print("\n📤 Upload a soybean leaf image:")
uploaded = files.upload()
img_path = list(uploaded.keys())[-1]

image = Image.open(img_path).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(image); plt.axis('off')
plt.title(f"Uploaded: {img_path}", fontsize=10)
plt.tight_layout(); plt.show()

# Preprocess & predict
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])
inp = transform(image).unsqueeze(0).to(device)
with torch.no_grad():
    out   = infer_model(inp)
    probs = F.softmax(out, dim=1)[0]
    top5_probs, top5_idx = torch.topk(probs, k=min(5, NUM_CLASSES))

pred_class  = CLASS_NAMES[top5_idx[0].item()]
confidence  = top5_probs[0].item()

print("\n" + "="*52)
print("  PREDICTION RESULTS")
print("="*52)
print(f"  Top prediction : {pred_class}")
print(f"  Confidence     : {confidence*100:.2f}%")
status = "✅ Healthy" if "healthy" in pred_class.lower() else "⚠️  Diseased"
print(f"  Status         : {status}")
print("\n  Top-5 predictions:")
for prob, idx in zip(top5_probs, top5_idx):
    name = CLASS_NAMES[idx.item()]
    bar  = "█" * int(prob.item() * 30)
    print(f"  {name:<40} {prob.item()*100:>5.1f}%  {bar}")
print("="*52)

# Bar chart
labels_p = [CLASS_NAMES[i].replace('_', '\n') for i in top5_idx]
vals_p   = [p.item()*100 for p in top5_probs]
cols_p   = ['#1D9E75' if 'healthy' in CLASS_NAMES[i].lower()
            else '#D85A30' for i in top5_idx]

plt.figure(figsize=(10, 4))
bars = plt.barh(labels_p[::-1], vals_p[::-1],
                color=cols_p[::-1], edgecolor='white')
plt.xlabel('Confidence (%)', fontsize=11)
plt.title(f'Top-5 Predictions — {best_fl_key.upper()}', fontsize=12)
plt.xlim(0, 110)
for bar, val in zip(bars, vals_p[::-1]):
    plt.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 22 — Health check
# ──────────────────────────────────────────────────────────────────
print("── Session Health Check ──────────────────────────────")
print("Libraries  :", "OK" if torch else "MISSING")
print("Dataset    :", "OK" if os.path.exists(DATA_DIR) else "NOT FOUND")
print("Device     :", device)
print("GPU        :", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "None — enable T4 GPU")
try:
    print("Variables  :", "OK" if CLASS_NAMES else "MISSING")
except NameError:
    print("Variables  : MISSING — run cells 2, 5, 6, 7, 8, 9, 10, 11")